# K-EmoCon: Ensemble Bagged Trees for Valence / Arousal
Replicates the **Ensemble Bagged Trees (BVP + EDA + HR + SKT)** setup using the Empatica E4 streams.

Run the cells top to bottom. **Only edit `ROOT` in the next cell.** It must be the folder that contains `e4_data/`, `metadata/`, `emotion_annotations/` and `neurosky_polar_data/`.

Three evaluation schemes are reported: pooled random k-fold (paper-style, leaky), per-subject blocked k-fold, and LOSO.

In [3]:
# ===== EDIT THIS =====
ROOT = r"C:\Users\User\Desktop\VS_CODE\Sahi\K EmoCon"# ====================

import warnings, numpy as np, pandas as pd
from pathlib import Path
from scipy import signal, stats
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings('ignore')
ROOT = Path(ROOT)

## Config

In [4]:
WINDOW_SEC = 10          # physiological context used per annotation (ends at annotation time)
ANNOT_STEP = 5           # annotations are every 5 s
LABEL_SOURCE = "self"    # "self" | "partner" | "external" (= aggregated external)
BINARIZE = "subject_mean"  # "subject_mean" (rating > participant mean) | "fixed3" (rating > 3)
USE_EEG = False          # add NeuroSky BrainWave band features (not part of the paper setup)
N_TREES = 30             # MATLAB "Bagged Trees" default
SEED = 42

# Annotation "seconds" are assumed to be measured from subjects.csv startTime. VERIFY this
# against the K-EmoCon paper. If your windows look misaligned, try "initTime".
ANNOT_ANCHOR = "startTime"

E4_FS = {"BVP": 64, "EDA": 4, "TEMP": 4, "HR": 1}

## Loading

In [5]:
def paths(root: Path):
    return dict(
        e4=root / "e4_data" / "e4_data",
        eeg=root / "neurosky_polar_data" / "neurosky_polar_data",
        subj=root / "metadata" / "metadata" / "subjects.csv",
        avail=root / "metadata" / "metadata" / "data_availability.csv",
        ann=root / "emotion_annotations" / "emotion_annotations",
    )


def annotation_file(P, pid, source):
    sub = {"self": ("self_annotations", "self"),
           "partner": ("partner_annotations", "partner"),
           "external": ("aggregated_external_annotations", "external")}[source]
    return P["ann"] / sub[0] / f"P{pid}.{sub[1]}.csv"


def load_annotations(fp: Path):
    """Read annotation CSV, mark 'x' as NaN, keep only seconds/arousal/valence."""
    df = pd.read_csv(fp, na_values=["x", "X", ""])
    df.columns = [c.strip() for c in df.columns]
    df = df[["seconds", "arousal", "valence"]].apply(pd.to_numeric, errors="coerce")
    return df.dropna(subset=["seconds", "arousal", "valence"]).reset_index(drop=True)


def load_stream(P, pid, name):
    fp = P["e4"] / str(pid) / f"E4_{name}.csv"
    if not fp.exists():
        return None
    df = pd.read_csv(fp)
    df = df.sort_values("timestamp").drop_duplicates("timestamp")
    return df

## Feature extraction

In [6]:
def _basic(x, p):
    if len(x) < 2:
        return {f"{p}_{k}": np.nan for k in
                ["mean", "std", "min", "max", "range", "skew", "kurt", "slope"]}
    t = np.arange(len(x))
    return {
        f"{p}_mean": np.mean(x), f"{p}_std": np.std(x),
        f"{p}_min": np.min(x), f"{p}_max": np.max(x),
        f"{p}_range": np.ptp(x),
        f"{p}_skew": stats.skew(x) if np.std(x) > 0 else 0.0,
        f"{p}_kurt": stats.kurtosis(x) if np.std(x) > 0 else 0.0,
        f"{p}_slope": np.polyfit(t, x, 1)[0],
    }


def _hrv(ibi_ms, p):
    keys = ["mean", "sdnn", "rmssd", "pnn50", "n"]
    out = {f"{p}_{k}": np.nan for k in keys}
    ibi_ms = ibi_ms[(ibi_ms > 300) & (ibi_ms < 2000)]  # physiologic filter
    out[f"{p}_n"] = len(ibi_ms)
    if len(ibi_ms) >= 3:
        d = np.diff(ibi_ms)
        out[f"{p}_mean"] = ibi_ms.mean()
        out[f"{p}_sdnn"] = ibi_ms.std()
        out[f"{p}_rmssd"] = np.sqrt(np.mean(d ** 2))
        out[f"{p}_pnn50"] = np.mean(np.abs(d) > 50)
    return out


def bvp_features(x, fs):
    f = _basic(x, "bvp")
    out = f
    if len(x) < fs * 3:
        out.update({"bvp_npeaks": np.nan, "bvp_amp_mean": np.nan, **_hrv(np.array([]), "bvp_ibi")})
        return out
    b, a = signal.butter(2, [0.5, 8], btype="band", fs=fs)
    xf = signal.filtfilt(b, a, x)
    peaks, props = signal.find_peaks(xf, distance=int(0.4 * fs), prominence=np.std(xf) * 0.5)
    out["bvp_npeaks"] = len(peaks)
    out["bvp_amp_mean"] = xf[peaks].mean() if len(peaks) else np.nan
    ibi = np.diff(peaks) / fs * 1000.0
    out.update(_hrv(ibi, "bvp_ibi"))
    return out


def eda_features(x, fs):
    out = _basic(x, "eda")
    if len(x) < fs * 3:
        out.update({"eda_tonic_mean": np.nan, "eda_phasic_std": np.nan,
                    "eda_scr_n": np.nan, "eda_scr_amp": np.nan})
        return out
    # tonic = low-pass (0.05 Hz), phasic = residual
    b, a = signal.butter(2, 0.05, btype="low", fs=fs)
    pad = min(len(x) - 1, 3 * max(len(a), len(b)))
    tonic = signal.filtfilt(b, a, x, padlen=pad)
    phasic = x - tonic
    peaks, _ = signal.find_peaks(phasic, prominence=0.01)
    out.update({"eda_tonic_mean": tonic.mean(), "eda_phasic_std": phasic.std(),
                "eda_scr_n": len(peaks),
                "eda_scr_amp": phasic[peaks].mean() if len(peaks) else 0.0})
    return out


def window_features(streams, t0, t1, use_eeg=False, eeg=None):
    """Extract features for absolute-time window [t0, t1) in ms."""
    feat = {}
    for name in ["BVP", "EDA", "HR", "TEMP"]:
        df = streams.get(name)
        if df is None:
            continue
        seg = df[(df.timestamp >= t0) & (df.timestamp < t1)]["value"].to_numpy(float)
        if name == "BVP":
            feat.update(bvp_features(seg, E4_FS["BVP"]))
        elif name == "EDA":
            feat.update(eda_features(seg, E4_FS["EDA"]))
        else:
            feat.update(_basic(seg, name.lower()))
    ibi = streams.get("IBI")
    if ibi is not None:
        seg = ibi[(ibi.timestamp >= t0) & (ibi.timestamp < t1)]["value"].to_numpy(float)
        feat.update(_hrv(seg, "e4ibi"))
    if use_eeg and eeg is not None:
        seg = eeg[(eeg.timestamp >= t0) & (eeg.timestamp < t1) & (eeg.isValid.astype(bool))]
        bands = ["delta", "theta", "lowAlpha", "highAlpha", "lowBeta",
                 "highBeta", "lowGamma", "middleGamma"]
        for b in bands:
            feat[f"eeg_{b}_mean"] = seg[b].mean() if len(seg) else np.nan
        feat["eeg_valid_n"] = len(seg)
    return feat

## Dataset build (alignment + windows)

In [7]:
def build_dataset(root: Path, label_source=LABEL_SOURCE, window_sec=WINDOW_SEC,
                  use_eeg=USE_EEG, anchor=ANNOT_ANCHOR):
    P = paths(root)
    subj = pd.read_csv(P["subj"]).set_index("pid")
    avail = pd.read_csv(P["avail"]).set_index("pid")
    rows = []
    for pid in subj.index:
        need = ["E4_BVP", "E4_EDA", "E4_HR", "E4_TEMP"]
        if not all(bool(avail.loc[pid, c]) for c in need if c in avail.columns):
            print(f"[skip] pid {pid}: missing E4 stream(s)")
            continue
        af = annotation_file(P, pid, label_source)
        if not af.exists():
            print(f"[skip] pid {pid}: no {label_source} annotation")
            continue
        streams = {n: load_stream(P, pid, n) for n in ["BVP", "EDA", "HR", "TEMP", "IBI"]}
        eeg = None
        if use_eeg:
            ef = P["eeg"] / str(pid) / "BrainWave.csv"
            eeg = pd.read_csv(ef) if ef.exists() else None
        ann = load_annotations(af)
        t_anchor = subj.loc[pid, anchor]
        for _, r in ann.iterrows():
            t_end = t_anchor + r["seconds"] * 1000.0
            t_start = t_end - window_sec * 1000.0
            f = window_features(streams, t_start, t_end, use_eeg, eeg)
            f.update(pid=int(pid), seconds=r["seconds"],
                     arousal=r["arousal"], valence=r["valence"])
            rows.append(f)
        print(f"[ok]   pid {pid}: {len(ann)} windows")
    df = pd.DataFrame(rows)
    return df


def add_labels(df, binarize=BINARIZE):
    df = df.copy()
    for t in ["arousal", "valence"]:
        if binarize == "subject_mean":
            thr = df.groupby("pid")[t].transform("mean")
            df[f"{t}_bin"] = (df[t] > thr).astype(int)
        else:
            df[f"{t}_bin"] = (df[t] > 3).astype(int)
    # 4-class quadrant: 0=LALV 1=LAHV 2=HALV 3=HAHV
    df["quadrant"] = df["arousal_bin"] * 2 + df["valence_bin"]
    return df

## Modeling and evaluation

In [8]:
def make_model():
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED),
        n_estimators=N_TREES, random_state=SEED, n_jobs=-1)


def fill_and_scale_fold(Xtr, Xte):
    """Median-impute using TRAIN statistics only (trees don't need scaling)."""
    med = np.nanmedian(Xtr, axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    Xtr = np.where(np.isnan(Xtr), med, Xtr)
    Xte = np.where(np.isnan(Xte), med, Xte)
    return Xtr, Xte


def _score(y, p):
    return dict(acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                f1=f1_score(y, p, average="macro"))


def eval_pooled_random(X, y, k=5):
    """Paper-style: pooled data, random stratified k-fold. LEAKY (neighbouring windows
    of the same subject land in train and test). Use only to compare with the paper."""
    yt, yp = [], []
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=SEED).split(X, y):
        Xtr, Xte = fill_and_scale_fold(X[tr], X[te])
        m = make_model().fit(Xtr, y[tr])
        yt += list(y[te]); yp += list(m.predict(Xte))
    return _score(np.array(yt), np.array(yp))


def eval_subject_dependent_blocked(X, y, pids, k=5):
    """Honest subject-dependent: per subject, contiguous (unshuffled) time folds so the
    test block is not interleaved with training windows. Scores are pooled over subjects."""
    yt, yp = [], []
    for pid in np.unique(pids):
        idx = np.where(pids == pid)[0]
        if len(idx) < k * 4 or len(np.unique(y[idx])) < 2:
            continue
        for tr, te in KFold(k, shuffle=False).split(idx):
            tr, te = idx[tr], idx[te]
            if len(np.unique(y[tr])) < 2:
                continue
            Xtr, Xte = fill_and_scale_fold(X[tr], X[te])
            m = make_model().fit(Xtr, y[tr])
            yt += list(y[te]); yp += list(m.predict(Xte))
    return _score(np.array(yt), np.array(yp))


def eval_loso(X, y, pids):
    """Subject-independent: Leave-One-Subject-Out."""
    yt, yp = [], []
    for tr, te in LeaveOneGroupOut().split(X, y, pids):
        Xtr, Xte = fill_and_scale_fold(X[tr], X[te])
        m = make_model().fit(Xtr, y[tr])
        yt += list(y[te]); yp += list(m.predict(Xte))
    return _score(np.array(yt), np.array(yp))


def run_experiments(df):
    id_cols = ["pid", "seconds", "arousal", "valence",
               "arousal_bin", "valence_bin", "quadrant"]
    feat_cols = [c for c in df.columns if c not in id_cols]
    # drop windows with (almost) no signal
    df = df[df[feat_cols].notna().mean(axis=1) > 0.5].reset_index(drop=True)
    X = df[feat_cols].to_numpy(float)
    pids = df["pid"].to_numpy()
    print(f"\n{len(df)} windows, {len(feat_cols)} features, {df.pid.nunique()} participants")

    results = []
    for task, col in [("Binary Arousal", "arousal_bin"),
                      ("Binary Valence", "valence_bin"),
                      ("4-Class Quadrant", "quadrant")]:
        y = df[col].to_numpy(int)
        print(f"\n=== {task}  (class balance: {np.bincount(y) / len(y)}) ===")
        for scheme, fn in [
            ("Pooled random 5-fold (paper-style, leaky)", lambda: eval_pooled_random(X, y)),
            ("Per-subject blocked 5-fold (subject-dependent)", lambda: eval_subject_dependent_blocked(X, y, pids)),
            ("LOSO (subject-independent)", lambda: eval_loso(X, y, pids)),
        ]:
            s = fn()
            print(f"  {scheme:52s} acc={s['acc']:.3f}  bal_acc={s['bal_acc']:.3f}  macroF1={s['f1']:.3f}")
            results.append(dict(task=task, scheme=scheme, **s))
    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #

## 1) Build the window-level feature table (cached to CSV)
Delete `kemocon_features.csv` if you change the window size, label source, anchor or `USE_EEG`.

In [9]:
CACHE = Path('kemocon_features.csv')
if CACHE.exists():
    df_raw = pd.read_csv(CACHE)
    print('Loaded cache:', df_raw.shape)
else:
    df_raw = build_dataset(ROOT, LABEL_SOURCE, WINDOW_SEC, USE_EEG, ANNOT_ANCHOR)
    df_raw.to_csv(CACHE, index=False)
    print('Built:', df_raw.shape)
df_raw.head()

[ok]   pid 1: 181 windows
[skip] pid 2: missing E4 stream(s)
[skip] pid 3: missing E4 stream(s)
[ok]   pid 4: 136 windows
[ok]   pid 5: 124 windows
[skip] pid 6: missing E4 stream(s)
[skip] pid 7: missing E4 stream(s)
[ok]   pid 8: 120 windows
[ok]   pid 9: 122 windows
[ok]   pid 10: 120 windows
[ok]   pid 11: 120 windows
[ok]   pid 12: 120 windows
[ok]   pid 13: 122 windows
[ok]   pid 14: 117 windows
[ok]   pid 15: 121 windows
[ok]   pid 16: 121 windows
[ok]   pid 17: 132 windows
[ok]   pid 18: 126 windows
[ok]   pid 19: 145 windows
[ok]   pid 20: 144 windows
[ok]   pid 21: 119 windows
[ok]   pid 22: 121 windows
[ok]   pid 23: 126 windows
[ok]   pid 24: 126 windows
[ok]   pid 25: 147 windows
[ok]   pid 26: 146 windows
[ok]   pid 27: 121 windows
[ok]   pid 28: 121 windows
[ok]   pid 29: 121 windows
[ok]   pid 30: 121 windows
[ok]   pid 31: 128 windows
[ok]   pid 32: 129 windows
Built: (3597, 52)


,bvp_mean,bvp_std,bvp_min,bvp_max,bvp_range,bvp_skew,bvp_kurt,bvp_slope,bvp_npeaks,bvp_amp_mean,...,temp_slope,e4ibi_mean,e4ibi_sdnn,e4ibi_rmssd,e4ibi_pnn50,e4ibi_n,pid,seconds,arousal,valence
0,0.415578,1.959224,-5.42,5.07,10.49,-0.297453,-0.093643,-0.000326,15.0,2.524249,...,0.00027,NaN,NaN,NaN,NaN,0.0,1,5,1,2
1,0.262672,2.149690,-7.56,6.29,13.85,-0.406777,0.714989,-0.001669,17.0,3.001043,...,0.00021,NaN,NaN,NaN,NaN,0.0,1,10,1,2
2,0.242438,2.334027,-7.56,6.29,13.85,-0.451286,0.482017,0.000935,15.0,3.229966,...,-0.00069,NaN,NaN,NaN,NaN,0.0,1,15,2,2
3,2.233219,14.875179,-96.33,144.24,240.57,6.028601,55.042422,0.017206,2.0,107.273663,...,0.00075,NaN,NaN,NaN,NaN,0.0,1,20,2,3
4,-0.197875,20.898683,-142.95,144.24,287.19,-0.198010,27.943544,-0.001057,2.0,78.978971,...,0.00003,NaN,NaN,NaN,NaN,0.0,1,25,2,3


## 2) Labels and sanity checks
Check that class balance is reasonable and that each participant has windows with real signal.

In [10]:
df = add_labels(df_raw, BINARIZE)
print(df.groupby('pid').size().describe())
print(df[['arousal_bin','valence_bin','quadrant']].apply(lambda s: s.value_counts(normalize=True)).round(3))

count     28.000000
mean     128.464286
std       13.628625
min      117.000000
25%      121.000000
50%      122.000000
75%      129.750000
max      181.000000
dtype: float64
   arousal_bin  valence_bin  quadrant
0         0.52        0.508     0.300
1         0.48        0.492     0.220
2          NaN          NaN     0.208
3          NaN          NaN     0.272


## 3) Run experiments
Expect the pooled random k-fold number to be highest (leakage). LOSO is the realistic figure.

In [11]:
results = run_experiments(df)
results.to_csv('kemocon_results.csv', index=False)
results.round(3)


3467 windows, 48 features, 28 participants

=== Binary Arousal  (class balance: [0.51860398 0.48139602]) ===
  Pooled random 5-fold (paper-style, leaky)            acc=0.796  bal_acc=0.794  macroF1=0.795
  Per-subject blocked 5-fold (subject-dependent)       acc=0.610  bal_acc=0.609  macroF1=0.609
  LOSO (subject-independent)                           acc=0.558  bal_acc=0.553  macroF1=0.549

=== Binary Valence  (class balance: [0.50360542 0.49639458]) ===
  Pooled random 5-fold (paper-style, leaky)            acc=0.855  bal_acc=0.855  macroF1=0.855
  Per-subject blocked 5-fold (subject-dependent)       acc=0.640  bal_acc=0.636  macroF1=0.636
  LOSO (subject-independent)                           acc=0.557  bal_acc=0.558  macroF1=0.555

=== 4-Class Quadrant  (class balance: [0.30025959 0.21834439 0.20334583 0.27805019]) ===
  Pooled random 5-fold (paper-style, leaky)            acc=0.753  bal_acc=0.747  macroF1=0.748
  Per-subject blocked 5-fold (subject-dependent)       acc=0.458  bal

,task,scheme,acc,bal_acc,f1
0,Binary Arousal,"Pooled random 5-fold (paper-style, leaky)",0.796,0.794,0.795
1,Binary Arousal,Per-subject blocked 5-fold (subject-dependent),0.610,0.609,0.609
2,Binary Arousal,LOSO (subject-independent),0.558,0.553,0.549
3,Binary Valence,"Pooled random 5-fold (paper-style, leaky)",0.855,0.855,0.855
4,Binary Valence,Per-subject blocked 5-fold (subject-dependent),0.640,0.636,0.636
5,Binary Valence,LOSO (subject-independent),0.557,0.558,0.555
6,4-Class Quadrant,"Pooled random 5-fold (paper-style, leaky)",0.753,0.747,0.748
7,4-Class Quadrant,Per-subject blocked 5-fold (subject-dependent),0.458,0.452,0.451
8,4-Class Quadrant,LOSO (subject-independent),0.289,0.267,0.261


## Notes
- Annotation `seconds` is assumed to count from `startTime` in `subjects.csv`. If results look like noise, set `ANNOT_ANCHOR = 'initTime'` and delete the cache.
- Try `BINARIZE = 'fixed3'` to compare against a fixed cutoff (expect class imbalance).
- Set `LABEL_SOURCE` to `'partner'` or `'external'` for other label sets.